# Extraction du catalogue FReSH depuis l'API vers excel agrégé

In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

In [4]:
BASE_SEARCH_URL = "https://nada.portail-fresh.fr/index.php/api/catalog/search"

def get_all_idnos(page_size=100):
    idnos = []
    page = 1

    while True:
        params = {
            "ps": page_size,
            "page": page
        }
        r = requests.get(BASE_SEARCH_URL, params=params)
        r.raise_for_status()
        data = r.json()

        rows = data["result"]["rows"]
        if not rows:
            break

        for row in rows:
            if row.get("idno"):
                idnos.append(row["idno"])

        if len(rows) < page_size:
            break

        page += 1

    return idnos


idnos = get_all_idnos()
print("Nombre total de fiches :", len(idnos))

Nombre total de fiches : 2154


In [5]:
DETAIL_URL = "https://nada.portail-fresh.fr/index.php/api/catalog"

HEADERS_XML = {
    "Accept": "application/xml"
}

def fetch_xml(idno):
    url = f"{DETAIL_URL}/{idno}"
    r = requests.get(url, headers=HEADERS_XML, timeout=30)
    r.raise_for_status()
    return idno, r.text

def flatten_xml(element, parent_key="", result=None):
    if result is None:
        result = {}

    for child in element.find_all(recursive=False):
        key = f"{parent_key}.{child.name}" if parent_key else child.name

        if child.find_all(recursive=False):
            flatten_xml(child, key, result)
        else:
            result[key] = child.get_text(strip=True)

    return result

def parse_study_xml(xml_text):
    soup = BeautifulSoup(xml_text, "xml")
    root = soup.find()  # premier élément racine
    return flatten_xml(root)

In [6]:
results = []

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {executor.submit(fetch_xml, idno): idno for idno in idnos}

    for future in as_completed(futures):
        idno = futures[future]
        try:
            _, xml = future.result()
            data = parse_study_xml(xml)
            data["idno"] = idno  # sécurité
            results.append(data)
            print(idno)
        except Exception as e:
            print("! ERREUR !", idno, e)

FRESH-PEF73274-en
FRESH-PEF3476-en
FRESH-PEF124-en
FRESH-PEF8214-en
FReSH-43661-en
FRESH-PEF1138-en
FRESH-PEF134-en
FRESH-PEF74138-en
FRESH-PEF1136-en
FRESH-PEF74126-en
FRESH-PEF74115-en
FRESH-PEF74087-en
FRESH-PEF79-en
FRESH-PEF73345-en
FRESH-PEF74127-en
FRESH-PEF3555-en
FRESH-PEF3096-en
FRESH-PEF74136-en
FRESH-PEF73840-en
FRESH-PEF73330-en
FRESH-PEF60148-en
FRESH-PEF151-en
FRESH-PEF74149-en
FRESH-PEF3594-en
FRESH-PEF74140-fr
FRESH-PEF60178-en
FRESH-PEF72-en
FRESH-PEF2295-fr
FRESH-PEF3666-en
FRESH-PEF73260-en
FRESH-PEF74135-en
FRESH-PEF7763-fr
FRESH-PEF60164-en
FRESH-PEF8293-en
FRESH-PEF148-en
FRESH-PEF8584-fr
FRESH-PEF3353-en
FRESH-PEF8584-en
FRESH-PEF73838-fr
FRESH-PEF60160-en
FRESH-PEF73303-en
FRESH-PEF38993-en
FRESH-PEF8214-fr
FRESH-PEF8593-en
FRESH-PEF3925-en
FRESH-PEF73375-en
FRESH-PEF73422-en
FRESH-PEF73894-fr
FRESH-PEF73886-fr
FRESH-PEF38925-en
FRESH-PEF5926-en
FRESH-PEF38979-en
FRESH-PEF73832-fr
FRESH-PEF60044-en
FRESH-PEF143-en
FRESH-PEF38905-en
FRESH-PEF6711-en
FRESH-PEF190

In [7]:
df = pd.DataFrame(results)
print(df.shape)
df.head()


(2154, 298)


,status,dataset.id,dataset.doi,dataset.repositoryid,dataset.type,dataset.idno,dataset.title,dataset.year_start,dataset.year_end,dataset.nation,...,dataset.metadata.additional.dataKind.values,dataset.metadata.study_desc.distribution_statement.contact.item.extlink,dataset.metadata.study_desc.study_info.topics,dataset.metadata.additional.interventionalStudy.researchPurpose.researchpurpose,dataset.metadata.additional.interventionalStudy.trialPhase.trialphase,dataset.metadata.additional.inclusionGroups.inclusiongroup.name,dataset.metadata.additional.inclusionGroups.inclusiongroup.description,dataset.metadata.additional.trialPhase.values.value,dataset.metadata.additional.universe.sexe,dataset.metadata.additional.sex.values
0,success,42900,,FReSH,survey,FRESH-PEF73274-en,,2015,2017,France,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,success,43620,,FReSH,survey,FRESH-PEF3476-en,"20,000 Volunteers for Medical Research",1989,2025,France,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,success,41516,,FReSH,survey,FRESH-PEF124-en,"A cross-sectional, multicentre, epidemiologica...",2006,2006,France,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,success,43378,,FReSH,survey,FRESH-PEF8214-en,"""Tumor Identity Cards (CIT)®"" Database",2000,2014,France,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,success,43662,,FReSH,survey,FReSH-43661-en,A database for in-depth phenotyping of craniof...,2026,2028,France,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
#Export
df.to_excel("catalogue_fresh_complet_xml_flat_20260921.xlsx", index=False)